In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events"


In [0]:
spark.read.format("delta").load(delta_path).count()


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F


In [0]:
spark.sql(f"""
DESCRIBE HISTORY delta.`{delta_path}`
""").show(truncate=False)


In [0]:
events_v0 = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path)

events_v0.count()


In [0]:
updates_df = spark.read \
    .format("delta") \
    .load(delta_path) \
    .limit(1000)

updates_df.show(5)
updates_df = spark.read \
    .format("delta") \
    .load(delta_path) \
    .limit(1000)

updates_df.show(5)


In [0]:
updates_df = updates_df.withColumn(
    "price", F.col("price") + 10
)


In [0]:
delta_table = DeltaTable.forPath(spark, delta_path)


In [0]:
delta_table.alias("t").merge(
    updates_df.alias("s"),
    "t.user_session = s.user_session AND t.event_time = s.event_time"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


In [0]:
spark.read.format("delta").load(delta_path).count()


In [0]:
%sql
CREATE OR REPLACE TABLE events_optimized
USING DELTA
AS
SELECT * FROM delta.`/Volumes/workspace/ecommerce/ecommerce_data/delta/events`


In [0]:
%sql
SHOW TABLES


In [0]:
%sql
OPTIMIZE events_optimized
ZORDER BY (event_type, user_id)


In [0]:
%sql
VACUUM events_optimized RETAIN 168 HOURS


In [0]:
%sql
OPTIMIZE delta.`/Volumes/workspace/ecommerce/ecommerce_data/delta/events`
ZORDER BY (event_type, user_id)


In [0]:
%sql
VACUUM delta.`/Volumes/workspace/ecommerce/ecommerce_data/delta/events` RETAIN 168 HOURS
